## National Sleep Research Resource (NSRR) Data Download

In this notebook a HEAL user can download data files from the [National Sleep Research Resource (NSRR)](https://sleepdata.org) using the NSRR API and a personal API token.

Before using this notebook, the user must:

* Have an NSRR account at [sleepdata.org](https://sleepdata.org). You can create one by visiting the site and signing up.
* Have submitted a data access request for the dataset(s) they wish to download and received approval.
* Have their NSRR API token, which can be found at [sleepdata.org/token](https://sleepdata.org/token) after logging in.

The only inputs required from the user are:

1. Your **NSRR API token** (entered securely via prompt in the cell below)
2. The **dataset slug** identifying the dataset to download (e.g., `cmmw` or `nop-nhp`)

**HEAL Study HDP00397** has deposited data at NSRR in the following datasets:
* [sleepdata.org/datasets/cmmw](https://sleepdata.org/datasets/cmmw)
* [sleepdata.org/datasets/nop-nhp](https://sleepdata.org/datasets/nop-nhp)

Please note: Users are responsible for complying with all aspects of their Data Use Agreement, including deleting any accessed data in accordance with the parameters specified in their DUA.

### Import Necessary Packages

In [ ]:
import requests
import getpass
from pathlib import Path

### Enter Your NSRR API Token

Your NSRR API token can be found at [sleepdata.org/token](https://sleepdata.org/token) after logging in. The token will be entered securely and will not be displayed.

In [ ]:
auth_token = getpass.getpass("Enter your NSRR API token: ")

### Define Dataset Slug

Enter the NSRR dataset slug for the dataset you wish to download. The slug is the identifier found in the dataset URL, e.g., for `https://sleepdata.org/datasets/cmmw` the slug is `cmmw`.

HEAL Study HDP00397 datasets: `cmmw`, `nop-nhp`

In [ ]:
dataset_slug = "nop-nhp"

### Test Authentication

Verify the API token is valid by requesting your NSRR account profile.

In [ ]:
NSRR_BASE_URL = "https://sleepdata.org"

response = requests.get(
    f"{NSRR_BASE_URL}/api/v1/account/profile.json",
    params={"auth_token": auth_token},
)

if response.status_code != requests.codes.ok:
    print("Failed to authenticate. Please check your API token.")
    response.raise_for_status()
else:
    profile = response.json()
    print(f"Authenticated as: {profile.get('email', 'unknown')}")

### Get Dataset Information

Retrieve and display metadata for the specified dataset.

In [ ]:
response = requests.get(
    f"{NSRR_BASE_URL}/api/v1/datasets/{dataset_slug}.json",
    params={"auth_token": auth_token},
)

if response.status_code != requests.codes.ok:
    print(f"Failed to retrieve dataset '{dataset_slug}'. Check that the slug is correct and you have access.")
    response.raise_for_status()
else:
    dataset_info = response.json()
    print(f"Dataset: {dataset_info.get('name', dataset_slug)}")
    print(f"Slug: {dataset_info.get('slug', dataset_slug)}")
    print(f"Description: {dataset_info.get('description', 'N/A')}")

### List All Files In Dataset

The NSRR files API returns files for a single directory level at a time. The cell below recursively walks the dataset directory tree to build a complete file listing.

In [ ]:
def list_files_recursive(dataset_slug, auth_token, path=""):
    """Recursively list all files in an NSRR dataset."""
    all_files = []
    response = requests.get(
        f"{NSRR_BASE_URL}/api/v1/datasets/{dataset_slug}/files.json",
        params={"auth_token": auth_token, "path": path},
    )
    if response.status_code != requests.codes.ok:
        print(f"Failed to list files at path '{path}': {response.status_code}")
        return all_files

    for item in response.json():
        if item.get("is_file"):
            all_files.append({
                "full_path": item.get("full_path", ""),
                "file_name": item.get("file_name", ""),
                "file_size": item.get("file_size", 0),
            })
        else:
            # It's a directory — recurse into it
            all_files.extend(
                list_files_recursive(dataset_slug, auth_token, path=item.get("full_path", ""))
            )
    return all_files

all_files = list_files_recursive(dataset_slug, auth_token)
print(f"Found {len(all_files)} files in dataset '{dataset_slug}':\n")
for f in all_files:
    size_mb = f['file_size'] / (1024 * 1024) if f['file_size'] else 0
    print(f"  {f['full_path']}  ({size_mb:.2f} MB)")

### Download All Files In The Dataset

Download all files listed above, preserving the dataset's directory structure under a local `downloads/` folder.

In [ ]:
def download_nsrr_file(dataset_slug, auth_token, file_path, download_dir="downloads"):
    """Download a single file from NSRR."""
    download_url = f"{NSRR_BASE_URL}/datasets/{dataset_slug}/files/a/{auth_token}/m/heal-platform-sdk/{file_path}"

    local_path = Path(download_dir) / file_path
    local_path.parent.mkdir(parents=True, exist_ok=True)

    response = requests.get(download_url, stream=True)
    if response.status_code != requests.codes.ok:
        print(f"  FAILED: {file_path} (HTTP {response.status_code})")
        return False

    with open(local_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    return True

print(f"Downloading {len(all_files)} files...\n")
success_count = 0
for file_info in all_files:
    file_path = file_info["full_path"]
    print(f"  Downloading: {file_path}")
    if download_nsrr_file(dataset_slug, auth_token, file_path):
        success_count += 1

print(f"\nCompleted: {success_count}/{len(all_files)} files downloaded to downloads/")

### Download Select Files By Path

To download only specific files, define a list of file paths from the listing above.

In [ ]:
select_file_paths = [
    "FFT/Fp2 - Oz/NHP_1241201_Dosing_11252019_NOPR agonist.txt",
]

print(f"Downloading {len(select_file_paths)} selected files...\n")
success_count = 0
for file_path in select_file_paths:
    print(f"  Downloading: {file_path}")
    if download_nsrr_file(dataset_slug, auth_token, file_path):
        success_count += 1

print(f"\nCompleted: {success_count}/{len(select_file_paths)} files downloaded to downloads/")